# EML Schrödinger Demonstration

This notebook demonstrates the EML transformation and validation workflow for the exp-log representation.

It includes:
- numeric equivalence checks between original and EML-transformed expressions
- explicit domain validation for original and transformed expressions
- weighted symbolic cost and circuit-size comparisons against an exp/log baseline


In [ ]:
import sys
from pathlib import Path

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.eml import (
    eml,
    exp_eml,
    log_eml,
    evaluate_eml,
    validate_log_domain,
    validate_transformed_domain,
    total_node_count,
    operator_count,
    eml_node_count,
    eml_depth,
    weighted_cost,
    symbolic_circuit_size,
)
from src.transform import transform, exp_log_rewrite
from src.schrodinger import x, original_schrodinger, eml_schrodinger

operator = original_schrodinger()
eml_operator = eml_schrodinger()

operator, eml_operator

## Numeric validation for the Schrödinger operator

In [ ]:
f_orig = sp.lambdify(x, operator, 'numpy')

eml_evaluated = evaluate_eml(eml_operator)
f_eml = sp.lambdify(
    x,
    eml_evaluated,
    [{"exp": np.exp, "log": np.log, "Abs": np.abs, "sqrt": np.sqrt}],
)

xs = np.linspace(-2.0, 2.0, 301)
y_orig = f_orig(xs)
y_eml = f_eml(xs)
error = np.abs(y_orig - y_eml)

print(f"Mean absolute error: {np.mean(error):.4e}")
print(f"Max absolute error: {np.max(error):.4e}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(xs, y_orig, label='original')
ax.plot(xs, y_eml, linestyle='--', label='EML evaluated')
ax.set_title('Schrödinger operator numeric comparison')
ax.legend()
plt.show()

## Domain validation examples

In [ ]:
examples = [
    {
        "name": "positive quadratic",
        "expr": x**2,
        "range": (-3.0, 3.0),
    },
    {
        "name": "square root",
        "expr": x**sp.Rational(1, 2),
        "range": (-1.0, 1.0),
    },
    {
        "name": "log(abs(x))",
        "expr": sp.log(sp.Abs(x)),
        "range": (-5.0, 5.0),
    },
]

for example in examples:
    expr = example["expr"]
    transformed = transform(expr)
    xs = np.linspace(example["range"][0], example["range"][1], 101)
    original_valid = validate_log_domain(expr, x, sample_points=xs)
    transformed_valid = validate_transformed_domain(transformed, x, sample_points=xs)

    print(f"{example['name']}")
    print("  original expression:", expr)
    print("  transformed expression:", transformed)
    print(f"  original log-domain valid: {original_valid}")
    print(f"  transformed domain valid: {transformed_valid}")
    print("  ---")


## Cost comparison with exp/log baseline


In [ ]:
expr = x * sp.exp(x) + sp.log(x + 2)
transformed_expr = transform(expr)
baseline_expr = exp_log_rewrite(expr)

print('Original expression:')
print(expr)
print('\nEML transformed:')
print(transformed_expr)
print('\nExp/log baseline:')
print(baseline_expr)

costs = {
    'original': weighted_cost(expr),
    'eml': weighted_cost(transformed_expr),
    'exp_log': weighted_cost(baseline_expr),
}
circuits = {
    'original': symbolic_circuit_size(expr),
    'eml': symbolic_circuit_size(transformed_expr),
    'exp_log': symbolic_circuit_size(baseline_expr),
}

print('\nWeighted costs:', costs)
print('Circuit sizes:', circuits)

labels = list(costs.keys())
weights = [costs[label] for label in labels]
size_vals = [circuits[label] for label in labels]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(labels, weights, color=['#4c72b0', '#dd8452', '#55a868'])
axes[0].set_title('Weighted symbolic cost')
axes[0].set_ylabel('Cost')

axes[1].bar(labels, size_vals, color=['#4c72b0', '#dd8452', '#55a868'])
axes[1].set_title('Symbolic circuit size')
axes[1].set_ylabel('Unique subexpression count')

plt.tight_layout()
plt.show()